# Avito NLP: кандидатогенерация для поиска услуг

Задача: для каждого запроса из `benchmark_queries` вернуть до 50 `item_id` из корпуса `benchmark_items` (189k объявлений).
Метрика: Recall@50, усреднённый по запросам. Порядок внутри топ-50 не важен.

### Структура ноутбука
* Часть 1. EDA: что за данные и какие решения из этого следуют.
* Часть 2. Бейзлайн: BM25 + приоры локации и подкатегории + фильтры. LB 0.8476. Ответ сохраняется в `answer_baseline.csv`.
* Часть 3. Финальная модель: пул из 500 кандидатов (бейзлайн + эмбеддинги USER2) и ранкер LightGBM. LB 0.9066. Ответ сохраняется в `answer.csv`. Финальная оценка качества находится здесь, на чистой валидации (см. 3.1).

Код моделей лежит в `src/`, ноутбук вызывает его и показывает результаты. Прогон целиком занимает около 25 минут на Apple M4 / 16 ГБ, если эмбеддинги уже лежат в `artifacts/` (они есть в репозитории). Без них добавляется около 1.5 часа на кодирование корпуса.

## Часть 1. EDA

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

DATA = 'data'
train = pd.read_parquet(f'{DATA}/train.parquet')
bq = pd.read_parquet(f'{DATA}/benchmark_queries.parquet')   # запросы без разметки
bi = pd.read_parquet(f'{DATA}/benchmark_items.parquet')     # корпус для поиска
import sys
sys.path.insert(0, 'src')
from data import check_inputs
check_inputs(train, bq, bi)       # структура входных файлов (см. README)
print('train', train.shape, '| bench queries', bq.shape, '| bench items', bi.shape)

Проверка входных данных пройдена. Строк train с объявлением из корпуса: 6.6%
train (497673, 19) | bench queries (2452, 6) | bench items (189212, 14)


In [2]:
def norm_text(s: str) -> str:
    """Простая нормализация: нижний регистр, ё в е, пунктуация в пробел."""
    s = str(s).lower().replace('ё', 'е')
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

train['q'] = train.search_query.map(norm_text)
bq['q'] = bq.search_query.map(norm_text)

# координаты хранятся строками/Decimal: переводим в float
for d in (train, bi):
    d['lat'] = pd.to_numeric(d.item_latitude, errors='coerce')
    d['lon'] = pd.to_numeric(d.item_longitude, errors='coerce')
    d['price'] = pd.to_numeric(d.item_price, errors='coerce')

train.head(3).T

,0,1,2
search_query,скупка телевизоров,автоподбор,баня на дровах
search_location_id,652430,640860,653240
search_is_delivery_search,0,0,0
search_infm_params_text,,Рейтинг пользователя 4 звезды и выше,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье"
search_category,114,114,114
item_title_raw,Скупка б/у техники,Автоподбор Разовый осмотр автомобиля,"Баня на дровах ""Прованс"" на Цветочной"
item_rating_reviews_count,91.0,462.0,NaN
item_rating,4.989011,4.982684,NaN
item_price,1.000000000000000,3500.000000000000000,1600.000000000000000
item_microcat_id,2303374,2303374,86469


### 1.1 Пропуски, категории, доставка

In [3]:
print('Доля пропусков в train:');  print(train.isna().mean()[lambda x: x > 0].round(3))
print('\nДоля пропусков в корпусе:'); print(bi.isna().mean()[lambda x: x > 0].round(3))
print('\nsearch_category (train):', train.search_category.value_counts().head(4).to_dict())
print('search_category (bench):', bq.search_category.value_counts().to_dict())
print('item_category_id (корпус):', bi.item_category_id.value_counts().head(5).to_dict())
print('\nДоля поисков с доставкой: train %.5f, bench %.5f' % (train.search_is_delivery_search.mean(), bq.search_is_delivery_search.mean()))

Доля пропусков в train:
item_rating_reviews_count    0.039
item_rating                  0.057
item_longitude               0.000
item_latitude                0.000
lat                          0.000
lon                          0.000
dtype: float64

Доля пропусков в корпусе:


item_rating_reviews_count    0.061
item_rating                  0.093
item_longitude               0.000
item_latitude                0.000
item_description_raw         0.000
lat                          0.000
lon                          0.000
dtype: float64

search_category (train): {114: 497635, 0: 34, 81: 2, 10: 2}
search_category (bench): {114: 2230, 0: 222}
item_category_id (корпус): {114: 187336, 19: 357, 112: 141, 40: 122, 33: 103}

Доля поисков с доставкой: train 0.00002, bench 0.00000


Выводы.
* Почти все объекты относятся к категории 114 ("услуги"). Категория как признак практически бесполезна.
* Рейтинг и число отзывов иногда пропущены (5-9%), это отдельный признак "нет отзывов".

### 1.2 Запросы: train и бенчмарк

In [4]:
vc = train.q.value_counts()
print('Уникальных запросов в train:', train.q.nunique(), '| уникальных item в train:', train.item_id.nunique())
print('Уникальных текстов в бенчмарке:', bq.q.nunique(), 'из', len(bq))
print('\nДлина запроса в словах:')
print('  train (по строкам)       %.2f' % train.q.str.split().str.len().mean())
print('  train (уникальные тексты) %.2f' % train.drop_duplicates('q').q.str.split().str.len().mean())
print('  bench                     %.2f' % bq.q.str.split().str.len().mean())
print('\nСколько раз текст запроса из бенчмарка встречается в train:')
print(bq.q.map(vc).fillna(0).describe(percentiles=[.5, .63, .75, .9]).round(1).to_dict())
print('Доля запросов бенчмарка, текст которых есть в train: %.3f' % bq.q.isin(set(train.q)).mean())
print('Доля уникальных train-запросов с частотой 1: %.3f' % (vc == 1).mean())
print('\nТоп запросов train:'); print(vc.head(10).to_dict())
print('\nСлучайные запросы бенчмарка:'); print(bq.search_query.sample(15, random_state=0).tolist())

Уникальных запросов в train: 73706 | уникальных item в train: 344825
Уникальных текстов в бенчмарке: 2451 из 2452

Длина запроса в словах:


  train (по строкам)       2.34


  train (уникальные тексты) 3.11
  bench                     3.20

Сколько раз текст запроса из бенчмарка встречается в train:
{'count': 2452.0, 'mean': 5.8, 'std': 36.5, 'min': 0.0, '50%': 0.0, '63%': 1.0, '75%': 2.0, '90%': 7.0, 'max': 1101.0}
Доля запросов бенчмарка, текст которых есть в train: 0.375
Доля уникальных train-запросов с частотой 1: 0.594

Топ запросов train:
{'маникюр': 6540, 'массаж': 5704, 'наращивание ресниц': 5398, 'установка кондиционеров': 4576, 'педикюр': 3436, 'сантехник': 3405, 'вывоз мусора': 3288, 'эвакуатор': 3263, 'электрик': 3158, 'автоэлектрик': 2790}

Случайные запросы бенчмарка:
['баки шугаринг', 'услуги мини экскаватора выравнивание земли сафонтьево истринский район', 'гелик аренда', 'ремонт разъемов', 'эцп без рутокена', 'штукатурно малярные работы', 'герметизация оконных швов', 'изготовление деревянных дверей под заказ', 'бухгалтерские услуги для ип и ооо', 'вывоз газовую плиту', 'замена ручек пластиковых окон', 'мини татуировки тату мастер тату моск

Выводы.
* Бенчмарк - это уникальные тексты запросов, и распределение у них хвостовое. Средняя длина (3.2 слова) совпадает со средней длиной уникальных запросов train (3.1), а не со средней по строкам (2.3). Похоже, бенчмарк сэмплирован равномерно по уникальным запросам.
* Только 37% текстов бенчмарка вообще есть в train, медианная частота равна 0. Простое "запоминание" запросов закроет лишь часть случаев, основная работа достаётся текстовому поиску.
* Встречаются опечатки ("ремон генераторов"), топонимы в запросе ("... сафонтьево истринский район"), бренды и сленг ("гелик", "биксипластия"). Нужна устойчивость к опечаткам, например, семантика.

### 1.3 Пересечение объявлений train и корпуса

In [5]:
tr_items, bi_items = set(train.item_id), set(bi.item_id)
print('Доля корпуса, встречавшаяся в train:       %.3f' % bi.item_id.isin(tr_items).mean())
print('Доля строк train с item из корпуса:        %.3f' % train.item_id.isin(bi_items).mean())
print('Дубликаты заголовков в корпусе (lower):    %.3f' % bi.item_title_raw.str.lower().duplicated().mean())
key = ['q', 'search_location_id', 'search_infm_params_text']
g = train.groupby(key).size()
print('\nПоисковых "сессий" (q, loc, фильтры) в train:', len(g), '| объявлений на сессию: mean %.2f, median %d' % (g.mean(), g.median()))

Доля корпуса, встречавшаяся в train:       0.096
Доля строк train с item из корпуса:        0.066


Дубликаты заголовков в корпусе (lower):    0.376



Поисковых "сессий" (q, loc, фильтры) в train: 354237 | объявлений на сессию: mean 1.40, median 1


Выводы.
* Корпус почти не пересекается с train: только 9.6% объявлений корпуса встречались в train. Коллаборативный сигнал ("популярность объявления", "кто выбирал это объявление") работает слабо. Train нужен для обучения связи между текстом запроса и текстом или подкатегорией объявления, а не конкретных id.
* В среднем на запрос выбирают 1.4 объявления, так что Recall@50 для одного запроса почти бинарный.
* 37% заголовков в корпусе дублируются ("Маникюр", "Электрик" и т. п.). Один только заголовок плохо различает объявления, решать будут локация и описание.

### 1.4 Локация как главный фильтр

In [6]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

print('Совпадение search_location_id == item_location_id в train: %.3f' % (train.search_location_id == train.item_location_id).mean())

# центроид каждой локации по координатам объявлений
all_items = pd.concat([train[['item_location_id', 'lat', 'lon']], bi[['item_location_id', 'lat', 'lon']]])
centroid = all_items.groupby('item_location_id')[['lat', 'lon']].median()
d = haversine(train.search_location_id.map(centroid.lat), train.search_location_id.map(centroid.lon), train.lat, train.lon)
print('Расстояние от центра локации поиска до объявления, км:', d.describe(percentiles=[.5, .9, .95, .99]).round(1).to_dict())

# локации поиска, в которых нет ни одного объявления: это регионы (Мособласть, Ленобласть и т. п.)
item_locs = set(all_items.item_location_id)
no_loc = train[~train.search_location_id.isin(item_locs)]
print('\nДоля строк train с "региональной" локацией поиска: %.3f' % (len(no_loc) / len(train)))
for l in no_loc.search_location_id.value_counts().head(3).index:
    s = no_loc[no_loc.search_location_id == l]
    print(f'  search_loc {l} (lat {s.lat.median():.2f}, lon {s.lon.median():.2f}) -> item_loc:',
          s.item_location_id.value_counts(normalize=True).head(4).round(2).to_dict())
b_no = ~bq.search_location_id.isin(set(bi.item_location_id))
print('\nВ бенчмарке таких запросов: %d (%.1f%%), и все их локации есть в train: %s' % (b_no.sum(), 100 * b_no.mean(), bq[b_no].search_location_id.isin(set(train.search_location_id)).all()))

Совпадение search_location_id == item_location_id в train: 0.831


Расстояние от центра локации поиска до объявления, км: {'count': 443928.0, 'mean': 19.2, 'std': 196.8, 'min': 0.0, '50%': 2.9, '90%': 13.4, '95%': 22.2, '99%': 232.6, 'max': 14773.1}

Доля строк train с "региональной" локацией поиска: 0.108
  search_loc 107620 (lat 55.75, lon 37.62) -> item_loc: {637640: 0.61, 638790: 0.03, 637770: 0.02, 638570: 0.02}
  search_loc 621540 (lat 55.75, lon 37.78) -> item_loc: {637640: 0.27, 653240: 0.11, 633540: 0.03, 650400: 0.02}
  search_loc 107621 (lat 59.93, lon 30.34) -> item_loc: {653240: 0.79, 636840: 0.04, 636736: 0.02, 636550: 0.01}



В бенчмарке таких запросов: 427 (17.4%), и все их локации есть в train: True


Локации иерархичны: например, `107620` означает Москву с областью, а объявления лежат в `637640` (Москва) и окрестных городах.

In [7]:
m = (train.groupby('search_location_id').item_location_id.value_counts(normalize=True)
       .rename('p').reset_index())
m = m[m.p >= 0.01]                      # отсекаем редкие "хвосты"
loc_map = m.groupby('search_location_id').item_location_id.apply(set)
loc_cnt = bi.item_location_id.value_counts()

def pool_size(loc):
    return sum(loc_cnt.get(x, 0) for x in loc_map.get(loc, {loc}))

bq['loc_pool'] = bq.search_location_id.map(pool_size)
print('Объявлений корпуса в "своих" локациях на запрос:', bq.loc_pool.describe(percentiles=[.1, .25, .5, .75]).round(0).to_dict())
print('Покрытие позитивов train этой картой: %.3f' % np.mean([
    il in loc_map.get(sl, {sl}) for sl, il in zip(train.search_location_id, train.item_location_id)]))

Объявлений корпуса в "своих" локациях на запрос: {'count': 2452.0, 'mean': 10136.0, 'std': 13990.0, 'min': 168.0, '10%': 849.0, '25%': 1422.0, '50%': 3157.0, '75%': 20119.0, 'max': 66388.0}


Покрытие позитивов train этой картой: 0.955


Выводы.
* В 83% случаев локация объявления совпадает с локацией поиска, в 95% объявление находится в пределах ~22 км. Остаток - "выездные" или удалённые услуги (99-й перцентиль около 230 км).
* 17% запросов бенчмарка заданы на уровне региона. Для них нужна карта "регион -> города" из train, одного прямого совпадения id недостаточно.
* Локация сужает пул с 189k до медианы ~3k объявлений на запрос. Это всё ещё в 60 раз больше 50, так что текстовая релевантность внутри локации - основная задача.
* Локацию лучше использовать как признак, а не как жёсткий фильтр: иначе теряем ~5-10% подходящих объектов.

### 1.5 Подкатегория объявления и фильтры запроса

In [8]:
qm = train.groupby('q').item_microcat_id.agg(lambda s: s.value_counts(normalize=True).iloc[0])
w = train.q.value_counts()[qm.index]
print('Средняя доля самой частой подкатегории в выборе по запросу (взвешенно): %.3f' % np.average(qm, weights=w))
print('Подкатегорий: в train %d, в корпусе %d; доля корпуса с подкатегорией не из train: %.3f' % (
    train.item_microcat_id.nunique(), bi.item_microcat_id.nunique(), (~bi.item_microcat_id.isin(set(train.item_microcat_id))).mean()))
print('\nПустые фильтры запроса: train %.2f, bench %.2f' % ((train.search_infm_params_text == '').mean(), (bq.search_infm_params_text == '').mean()))
print(bq.search_infm_params_text.value_counts().head(8))

Средняя доля самой частой подкатегории в выборе по запросу (взвешенно): 0.863
Подкатегорий: в train 212, в корпусе 752; доля корпуса с подкатегорией не из train: 0.009

Пустые фильтры запроса: train 0.33, bench 0.63
search_infm_params_text
                                         1546
Вид услуги                                 60
Вид услуги Оборудование, производство      53
Вид услуги Ремонт и отделка                48
Вид услуги Праздники, мероприятия          40
Вид услуги Обучение, курсы                 39
Вид услуги Красота, здоровье               36
Вид услуги Вывоз мусора и вторсырья        36
Name: count, dtype: int64


In [9]:
# Согласуются ли фильтры запроса ("Вид услуги X") с параметрами объявления?
def vid(s):
    m_ = re.search(r'Вид услуги ([^А-ЯЁ]*[А-ЯЁ][^А-ЯЁ]*?)(?= [А-ЯЁ]|$)', s or '')
    return m_.group(1).strip() if m_ else None
t = train[train.search_infm_params_text.str.contains('Вид услуги [А-ЯЁ]', regex=True)].sample(20000, random_state=0)
sv, iv = t.search_infm_params_text.map(vid), t.item_infm_params_text.map(vid)
print('Совпадает "Вид услуги" запроса и объявления: %.3f' % (sv == iv).mean())

Совпадает "Вид услуги" запроса и объявления: 0.984


Выводы.
* Запрос почти однозначно задает подкатегорию: в среднем 86% выборов по запросу приходится на одну подкатегорию. Классификатор "текст запроса -> microcat" станет сильным признаком для ранкера и источником кандидатов.
* В 37% запросов бенчмарка есть фильтр "Вид услуги ...", он почти всегда совпадает с параметром объявления. Это надёжное почти жесткое ограничение.

### 1.6 Лексическое пересечение запроса и объявления

In [10]:
def toks(s):
    # грубый "стемминг": первые 5 символов слова, слова короче 3 символов отбрасываем
    return {w[:5] for w in re.findall(r'\w+', str(s).lower().replace('ё', 'е')) if len(w) > 2}

s = train.sample(30000, random_state=0)
qt = s.search_query.map(toks)
fields = {
    'title': s.item_title_raw,
    'title+params': s.item_title_raw + ' ' + s.item_infm_params_text.fillna(''),
    'title+params+desc': s.item_title_raw + ' ' + s.item_infm_params_text.fillna('') + ' ' + s.item_description_raw.fillna(''),
}
rows = []
for name, col in fields.items():
    ct = col.map(toks)
    rows.append((name, np.mean([len(a & b) > 0 for a, b in zip(qt, ct)]), np.mean([a <= b for a, b in zip(qt, ct)])))
print(pd.DataFrame(rows, columns=['поля', 'хоть одно слово', 'все слова']).round(3))
print('\nДлина описания, символов:', bi.item_description_raw.str.len().describe().round(0).to_dict())

                поля  хоть одно слово  все слова
0              title            0.835      0.487
1       title+params            0.915      0.620
2  title+params+desc            0.973      0.833

Длина описания, символов: {'count': 189177.0, 'mean': 1405.0, 'std': 1284.0, 'min': 1.0, '25%': 422.0, '50%': 1054.0, '75%': 1978.0, 'max': 8494.0}


Выводы.
* Лексический сигнал сильный: в 97% пар хотя бы одно слово запроса есть в тексте объявления, в 83% есть все слова. Но почти для половины пар все слова находятся только с учётом описания. Описание обязательно индексировать (BM25).
* Остальные ~3-15% - синонимы и перефразы, их закрывает dense-ретривер.
* Описания длинные (медиана ~1000 символов). Для энкодера нужно обрезать их или брать заголовок + параметры + начало описания.

## Итоги EDA и план решения

* Бенчмарк состоит из уникальных хвостовых запросов, только 37% из них встречались в train. Поэтому валидацию строим по сессиям, выбранным равномерно по уникальным текстам, и делаем упор на текстовый поиск, а не на запоминание запросов.
* Корпус почти не пересекается с train по item_id. Train используем для обучения связи между текстом запроса и текстом объявления, а также между запросом и подкатегорией.
* Локация сужает пул до ~3k объявлений, 17% запросов заданы на уровне региона. Нужна карта "регион -> города" из train и мягкий учёт локации и расстояния.
* Запрос почти однозначно задаёт подкатегорию. Классификатор "запрос -> microcat" пригодится как источник кандидатов и как признак.
* Лексический сигнал сильный, и описания важны. Используем BM25 по заголовку, параметрам и описанию, а против опечаток символьные n-граммы.
* Для перефраз и сленга нужен dense-ретривер на эмбеддингах.

# Часть 2. Бейзлайн

## 2.1 Локальная валидация

Повторяем устройство бенчмарка:
* Поисковая сессия задаётся тройкой `(нормализованный запрос, локация, фильтры)`, её позитивы - все выбранные в ней объявления.
* Сэмплируем 3000 уникальных текстов запросов равномерно (как в бенчмарке, где все тексты уникальны) и для каждого берём одну случайную сессию.
* Строки этих сессий убираются из обучения. Другие сессии с тем же текстом остаются, как и в реальности: 37% текстов бенчмарка встречались в train.
* Корпус: `benchmark_items` плюс позитивы валидации. Искать нужно среди тех же 189k реальных дистракторов.

In [11]:
import time, dataclasses
from common import recall_at_k, parse_filters
from baseline import Params, BaselineRetriever

KEY = ['q', 'search_location_id', 'search_infm_params_text']
rng = np.random.default_rng(42)
sessions = train[KEY].drop_duplicates()
n_val = min(3000, sessions.q.nunique() // 4)
val_texts = set(rng.choice(sessions.q.unique(), n_val, replace=False))
val_sessions = sessions[sessions.q.isin(val_texts)].sample(frac=1, random_state=42).drop_duplicates('q')

is_val = train[KEY].merge(val_sessions.assign(_v=1), on=KEY, how='left')._v.notna().values
train_fit, val_rows = train[~is_val], train[is_val]

# одна строка на валидационный запрос + множество его позитивов
vq = (val_rows.groupby(KEY, sort=False)
      .agg(search_query=('search_query', 'first'), item_ids=('item_id', set)).reset_index())
extra = val_rows.drop_duplicates('item_id')[bi.columns.drop(['lat', 'lon', 'price'], errors='ignore')]
extra = extra[~extra.item_id.isin(set(bi.item_id))]
val_corpus = pd.concat([bi[extra.columns], extra], ignore_index=True)

print('валидационных запросов:', len(vq), '| позитивов на запрос: %.2f' % vq.item_ids.map(len).mean())
BENCH_SEEN = bq.q.isin(set(train.q)).mean()     # доля запросов бенчмарка, чей текст есть в train
print('доля текстов, встречающихся в train_fit: %.3f (в бенчмарке %.3f)' % (vq.q.isin(set(train_fit.q)).mean(), BENCH_SEEN))
print('корпус валидации:', len(val_corpus))

валидационных запросов: 3000 | позитивов на запрос: 1.06
доля текстов, встречающихся в train_fit: 0.377 (в бенчмарке 0.375)
корпус валидации: 192125


## 2.2 Модель

Код лежит в `src/common.py` (токенизация, BM25, метрика) и `src/baseline.py` (ретривер). Скор объявления - линейная комбинация четырёх сигналов, и каждый из них отвечает на одну из находок EDA:

* BM25 по заголовку (вес 3), параметрам (вес 1) и описанию (вес 2), стемминг Snowball. Даёт текстовую релевантность. Описание нужно, потому что без него все слова запроса находятся только в 62% пар, а с ним в 83%.
* Лог-приор локации `log p(item_loc | search_loc)` по train. Поднимает объявления "своего" города и переводит регион в список городов. Основание: 17% поисков заданы по региону, а в 95% случаев объявление находится в пределах 22 км.
* Лог-приор подкатегории `log p(microcat | q)`: kNN по символьным n-граммам запросов train. Помогает найти правильный тип услуги даже без точного совпадения слов, потому что запрос на 86% определяет подкатегорию.
* Штраф за фильтры: доля фраз "Вид/Тип услуги ..." из запроса, найденных в параметрах объявления. Так учитываем фильтры, выбранные пользователем: они совпадают с параметрами объявления в 94-98% случаев.

Веса (`a=0.1, c=0.03, f=0.3`, BM25 `k1=1.2, b=0.75`, веса полей) подобраны небольшой сеткой на локальной валидации. Разброс между соседними точками сетки около 0.001-0.003, так что результат к ним нечувствителен.

### 2.3 Абляция на валидации

In [12]:
t0 = time.time()
ablation = {
    'BM25':                         dict(a_loc=0, c_mc=0, f_filter=0),
    '+ приор локации':              dict(c_mc=0, f_filter=0),
    '+ приор подкатегории':         dict(f_filter=0),
    '+ фильтры запроса (бейзлайн)': dict(),
}
gold = vq.item_ids.tolist()
val_scores = {}
for name, kw in ablation.items():
    r = BaselineRetriever(dataclasses.replace(Params(), **kw)).fit(train_fit, val_corpus)
    val_scores[name] = recall_at_k(r.predict(vq), gold)
    print(f'{name:32s} Recall@50 = {val_scores[name].mean():.4f}   ({time.time() - t0:.0f} c)')

BM25                             Recall@50 = 0.4060   (65 c)


+ приор локации                  Recall@50 = 0.8321   (126 c)


+ приор подкатегории             Recall@50 = 0.8692   (188 c)


+ фильтры запроса (бейзлайн)     Recall@50 = 0.8817   (267 c)


### 2.4 Где бейзлайн ошибается

In [13]:
vq['recall'] = val_scores['+ фильтры запроса (бейзлайн)']
vq['текст встречался в train'] = vq.q.isin(set(train_fit.q))
vq['есть фильтры'] = vq.search_infm_params_text.map(lambda s: len(parse_filters(s)) > 0)
vq['слов в запросе'] = vq.q.str.split().str.len().clip(upper=5)
for c in ['текст встречался в train', 'есть фильтры', 'слов в запросе']:
    print(vq.groupby(c).recall.agg(['mean', 'size']).round(3), '\n')

# ранг пропущенных позитивов под итоговым скором: насколько далеко они от топ-50
pos = pd.Series(np.arange(len(r.items)), index=r.items.item_id)
miss = vq[vq.recall < 1]
rows = []
for s in range(0, len(miss), 256):
    qb = miss.iloc[s:s + 256]
    S = r.score_batch(qb)
    for i, (g, sl) in enumerate(zip(qb.item_ids, qb.search_location_id)):
        for it in g:
            j = pos[it]
            rows.append(dict(query=qb.search_query.iloc[i], rank=int((S[i] > S[i, j]).sum()),
                             same_loc=r.items.item_location_id[j] == sl, item_title=r.items.item_title_raw[j]))
err = pd.DataFrame(rows)
print('Пропущено позитивов:', len(err))
print('Ранг пропущенного позитива (квантили):', err['rank'].quantile([.25, .5, .75, .9]).to_dict())
print('Доля пропусков, где объявление в той же локации: %.2f' % err.same_loc.mean())
err.sample(15, random_state=0)

                           mean  size
текст встречался в train             
False                     0.868  1869
True                      0.904  1131 

               mean  size
есть фильтры             
False         0.869  1470
True          0.894  1530 

                 mean  size
слов в запросе             
1               0.858   123
2               0.889   902
3               0.878   970
4               0.888   628
5               0.871   377 



Пропущено позитивов: 416
Ранг пропущенного позитива (квантили): {0.25: 70.0, 0.5: 147.5, 0.75: 590.75, 0.9: 2892.0}
Доля пропусков, где объявление в той же локации: 0.40


,query,rank,same_loc,item_title
358,набивка вин номера,10325,False,Изготовление шильдов табличек наклеек
170,репетитор по корейскому,296,False,Репетитор корейского языка
224,световые линии,37,True,Освещение+Карнизы+Монтаж под ключ
389,авторазбор иномарок,53,False,Помощь в Сервисе и в Подборе
308,рено логан под выкуп,67,False,Аренда автомобиля рено логан Renault Logan
307,ушив джинс,64,True,"Пошив, ремонт и подгонка одежды"
150,модульный дом 90 м2 под ключ,539,False,Каркасный дом 64 м2 для постоянного проживания
10,монтаж секционных ворот,635,False,Монтаж и ремонт ворот и автоматики и обслуживание
21,коворкинг для керамистов,53,True,Коворкинг керамика. Гончарка
261,ремонт напольной плитки,85,True,"Ремонт плитки, паркета, реставрация"


Выводы по бейзлайну.
* Главный вклад даёт локация (recall растёт с 0.41 до 0.83), дальше подкатегория (+3.7 п.п.) и фильтры (+1.2 п.п.).
* Запросы, которых не было в train, заметно хуже (0.87 против 0.90). Семантика для новых формулировок - главное направление роста.
* 60% пропусков - объявления из другой локации (выездные мастера, соседние города, онлайн-услуги). Лог-приор локации для них слишком жёсткий, нужны признаки расстояния и "удалённого" формата работы.
* Медианный ранг пропущенного позитива около 150. Если расширить пул кандидатов до 500-1000 и переранжировать его обученной моделью с признаками и dense-ретривером, большую часть этих случаев можно поднять в топ-50. Это следующий шаг.

## 2.5 Ответ на бенчмарке (посылка 1: LB 0.8476)

Для итогового ответа статистики (приоры локации и подкатегории) считаются по всему train, а индекс строится по `benchmark_items`.

In [14]:
t0 = time.time()
final = BaselineRetriever(Params()).fit(train, bi[extra.columns])
predictions = final.predict(bq)
print('готово за %.0f c' % (time.time() - t0))

answer = pd.DataFrame({'query_id': bq.query_id.values,
                       'answer': [' '.join(top50) for top50 in predictions]})
answer.to_csv('answer_baseline.csv', index=False)

# проверка формата по требованиям задания
chk = pd.read_csv('answer_baseline.csv', dtype=str)
lists = chk.answer.str.split(' ')
corpus_ids = set(bi.item_id)
assert list(chk.columns) == ['query_id', 'answer']
assert len(chk) == len(bq) and set(chk.query_id) == set(bq.query_id) and chk.query_id.is_unique
assert chk.query_id.str.len().eq(16).all()
assert lists.map(len).le(50).all() and lists.map(lambda l: len(l) == len(set(l))).all()
assert lists.map(lambda l: all(x in corpus_ids for x in l)).all()
print('answer_baseline.csv: формат корректен;', len(chk), 'строк; кандидатов в строке:', lists.map(len).min(), '-', lists.map(len).max())
chk.head(3)

готово за 66 c
answer_baseline.csv: формат корректен; 2452 строк; кандидатов в строке: 50 - 50


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf d722bcda1a555091 255fbeaf526a1cc1 4494aa11cbdc2d65 deb58d7a...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 1c6d2079e07f4497 367af128a9ea2a48 431efbc8...
2,LZCZNoVG4AFUkVRJ,168a9207e80b0be4 3b370cc603f67947 dab52187b4500d9b 3f89b8062dc85f1c 10eb6d6d...


# Часть 3. Двухэтапная модель: широкий пул кандидатов, dense-ретривер и ранкер

Бейзлайн на лидерборде: 0.8476.

Схема:
1. Этап 1, пул из 500 кандидатов. Скор бейзлайна (BM25 + приоры) плюс косинус эмбеддингов `deepvk/USER2-base`. Полнота пула около 0.98-0.99.
2. Этап 2, LightGBM LambdaRank. Ранкер по ~40 признакам пары "запрос - объявление" переупорядочивает пул, в ответ идёт топ-50.

In [15]:
import gc
from data import ITEM_COLS, split_in_corpus
from dense import item_embeddings, query_embeddings, EMB_DIM
from features import FeatureBuilder
from ranker import fit_ranker, feature_names, recall_from_features, top_k, LGB_PARAMS

# освобождаем объекты части 2 (индексы бейзлайна занимают несколько ГБ) и переиспользуем загруженные данные
del r, final, val_corpus, train_fit, val_rows
gc.collect()
train3, bq3, bi3 = train, bq, bi[ITEM_COLS].copy()
stats_c, rank_q, val_q = split_in_corpus(train3, bi3)
clean_q = pd.concat([val_q, rank_q], ignore_index=True)       # 11k запросов для CV ранкера
n_pos = clean_q.item_ids.map(len).values
seen = clean_q.q.isin(set(stats_c.q)).values
print('stats:', len(stats_c), 'строк | чистых запросов:', len(clean_q),
      '| доля знакомых текстов: %.2f (в бенчмарке %.2f)' % (seen.mean(), BENCH_SEEN))

def report(r):
    """Recall@50: как есть и перевзвешенный под долю знакомых/новых запросов бенчмарка."""
    return f'{r.mean():.4f} (под бенчмарк: {BENCH_SEEN * r[seen].mean() + (1 - BENCH_SEEN) * r[~seen].mean():.4f})'

stats: 485734 строк | чистых запросов: 11155 | доля знакомых текстов: 0.71 (в бенчмарке 0.37)


## 3.2 Эмбеддинги USER2-base

`deepvk/USER2-base` - открытая русскоязычная эмбеддинг-модель (ModernBERT, 150M параметров), работает локально. Объявление кодируется как заголовок плюс начало описания (до 128 токенов) с префиксом `search_document:`, запрос кодируется с префиксом `search_query:`.

Размерность 128 вместо 768. Модель обучена с Matryoshka (MRL): первые k координат эмбеддинга сами являются эмбеддингом. Замеры на чистой валидации:
* 768 измерений (291 МБ): только эмбеддинги дают recall@50 0.343, в смеси с бейзлайном 0.915 @50 и 0.987 @500;
* 256 измерений (97 МБ): 0.344, в смеси 0.915 и 0.987;
* 128 измерений (48 МБ): 0.337, в смеси 0.916 и 0.987;
* 64 измерения (24 МБ): 0.316, в смеси 0.916 и 0.987.

В смеси с бейзлайном обрезка не влияет на качество, поэтому берём 128 измерений. Файл эмбеддингов корпуса уменьшается с 290 до 47 МБ и помещается в репозиторий.

Артефакты. Эмбеддинги корпуса и запросов лежат в `artifacts/` и загружаются оттуда, если совпадают `item_id` или тексты запросов. Иначе они пересчитываются: корпус на Apple M4 (MPS) кодируется около 1.5 часа, модель скачивается с HuggingFace при первом запуске.

In [16]:
t0 = time.time()
item_emb = item_embeddings(bi3)                                                   # artifacts/emb_items_user2_128.npz
clean_q_emb = query_embeddings(clean_q.search_query, 'artifacts/emb_q_clean_user2_128.npz')
bench_q_emb = query_embeddings(bq3.search_query, 'artifacts/emb_q_bench_user2_128.npz')
print('эмбеддинги:', item_emb.shape, '| размерность', EMB_DIM, '| %.0f c' % (time.time() - t0))

эмбеддинги: (189212, 128) | размерность 128 | 0 c


## 3.3 Этап 1: пул кандидатов

Скор пула: скор бейзлайна плюс `1.0 * cos`. Вклад эмбеддингов в полноту пула:
* только эмбеддинги: 0.344 @50, 0.558 @200, 0.710 @500;
* эмбеддинги + приор локации: 0.808 @50, 0.886 @200, 0.916 @500;
* бейзлайн: 0.910 @50, 0.965 @200, 0.981 @500;
* бейзлайн + 1.0 * cos: 0.915 @50, 0.972 @200, 0.987 @500.

Сами по себе эмбеддинги без дообучения слабее BM25, но хорошо его дополняют: пул теряет в 1.5 раза меньше позитивов. Вес 1.0 оказался лучшим из проверенных (от 0.1 до 1.0).

In [17]:
t0 = time.time()
N_CAND, DENSE_W = 500, 1.0
ret_c = BaselineRetriever(Params()).fit(stats_c, bi3)
cand, cand_s = ret_c.candidates(clean_q, N_CAND, clean_q_emb, item_emb, dense_w=DENSE_W)
ids3 = bi3.item_id.values
order = np.argsort(-cand_s, axis=1)
cand, cand_s = np.take_along_axis(cand, order, 1), np.take_along_axis(cand_s, order, 1)
for n in (50, 200, 500):
    print(f'полнота пула @{n}:', report(recall_at_k([list(ids3[row[:n]]) for row in cand], clean_q.item_ids.tolist(), n)))
print('%.0f c' % (time.time() - t0))

полнота пула @50: 0.9158 (под бенчмарк: 0.9038)
полнота пула @200: 0.9727 (под бенчмарк: 0.9688)


полнота пула @500: 0.9872 (под бенчмарк: 0.9846)
149 c


## 3.4 Признаки и ранкер

Признаки (`src/features.py`):
* этап 1: скор пула и ранг по нему;
* текст: BM25 по заголовку, параметрам и описанию отдельно; доля слов запроса в заголовке и во всём тексте; символьное сходство запроса с заголовком (устойчиво к опечаткам); точное вхождение фразы;
* локация: лог-приор `p(item_loc | search_loc)`, совпадение id, расстояние до центра локации поиска в км и ранг по расстоянию, плотность объявлений в локации и в паре "подкатегория и локация";
* подкатегория: `p(microcat | q)` и её ранг;
* фильтры: доля фраз "Вид/Тип услуги", найденных в параметрах;
* объявление: рейтинг, отзывы, цена, выезд или удалённая работа, длины текстов, число дублей заголовка, флаги;
* запрос: длина, встречался ли текст в train, сходство с ближайшим запросом train, регион или город;
* dense: косинус эмбеддингов и ранг по нему.

In [18]:
t0 = time.time()
fb_c = FeatureBuilder(ret_c, stats_c, item_emb=item_emb)
F = fb_c.build(clean_q, cand, cand_s, q_emb=clean_q_emb)
pos_idx = pd.Series(np.arange(len(bi3)), index=bi3.item_id)
pos_sets = [set(pos_idx[list(g)]) for g in clean_q.item_ids]
F['label'] = [int(d in pos_sets[q]) for q, d in zip(F.qid.values, F.doc.values)]
feats = feature_names(F)
print('признаков:', len(feats), '| строк:', len(F), '| %.0f c' % (time.time() - t0))

признаков: 38 | строк: 5577500 | 79 c


In [19]:
# 5-fold CV по запросам: каждый запрос оценивается моделью, не видевшей его при обучении
t0 = time.time()
fold = np.random.default_rng(0).integers(0, 5, len(clean_q))
cv_score = np.zeros(len(F), np.float32)
for k in range(5):
    te = fold[F.qid.values] == k
    cv_score[te] = fit_ranker(F[~te], feats).predict(F.loc[te, feats])
dense_feats = ['dense_cos', 'dense_rank']
cv_score_nodense = np.zeros(len(F), np.float32)
for k in range(5):
    te = fold[F.qid.values] == k
    fn = [f for f in feats if f not in dense_feats]
    cv_score_nodense[te] = fit_ranker(F[~te], fn).predict(F.loc[te, fn])

print('Этап 1, топ-50 по скору пула:  ', report(recall_from_features(F, F.s1_score.values, n_pos)))
print('Ранкер без dense-признаков:     ', report(recall_from_features(F, cv_score_nodense, n_pos)))
print('Ранкер (все признаки):          ', report(recall_from_features(F, cv_score, n_pos)))
print('%.0f c' % (time.time() - t0))

Этап 1, топ-50 по скору пула:   0.9158 (под бенчмарк: 0.9038)


Ранкер без dense-признаков:      0.9475 (под бенчмарк: 0.9392)


Ранкер (все признаки):           0.9510 (под бенчмарк: 0.9441)
700 c


In [20]:
final_ranker = fit_ranker(F, feats)            # итоговый ранкер обучается на всех 11k запросах
imp = pd.Series(final_ranker.booster_.feature_importance('gain'), index=feats)
(imp / imp.sum()).sort_values(ascending=False).head(20).round(3)

s1_rank               0.510
s1_score              0.130
dist_log              0.053
cover_all             0.032
mc_loc_density_log    0.030
bm25_rel              0.027
dense_rank            0.022
loc_logp              0.020
dense_cos             0.017
reviews_log           0.016
mc_p                  0.012
desc_len_log          0.012
bm25_desc             0.011
dist_rank             0.011
rating                0.010
q_loc_density_log     0.010
price_log             0.009
bm25_title            0.009
title_char_cos        0.009
bm25_params           0.008
dtype: float64

Сводка на чистой валидации (5-fold CV, 11k запросов). В скобках recall, перевзвешенный под бенчмарк.
* Бейзлайн (BM25 + приоры), LB 0.8476: 0.910 (0.895).
* Плюс ранкер: 0.941 (0.931).
* Плюс эмбеддинги в пуле кандидатов: 0.948 (0.939).
* Плюс dense-признаки в ранкере: 0.951 (0.944).

Важнее всего для ранкера скор и ранг этапа 1, дальше идут покрытие слов запроса, расстояние и конкуренция в локации.

## 3.5 Ответ на бенчмарке (посылка 2)

* Статистики (приоры локации и подкатегории, признак "знакомости" запроса) считаем по всему train: для бенчмарка это честная история.
* Индекс и эмбеддинги строим по `benchmark_items`, пул из 500 кандидатов тем же этапом 1, ранкер обучен выше на 11k чистых запросах.

In [21]:
t0 = time.time()
ret_f = BaselineRetriever(Params()).fit(train3, bi3)
fb_f = FeatureBuilder(ret_f, train3, item_emb=item_emb)
cand_b, cand_bs = ret_f.candidates(bq3, N_CAND, bench_q_emb, item_emb, dense_w=DENSE_W)
Fb = fb_f.build(bq3, cand_b, cand_bs, q_emb=bench_q_emb)
top = top_k(Fb, final_ranker.predict(Fb[feats]), 50)
top['item_id'] = ids3[top.doc.values]
pred = top.groupby('qid').item_id.apply(list).reindex(range(len(bq3)))
print('готово за %.0f c' % (time.time() - t0))

answer = pd.DataFrame({'query_id': bq3.query_id.values, 'answer': [' '.join(p) for p in pred]})
answer.to_csv('answer.csv', index=False)

# проверка формата
chk = pd.read_csv('answer.csv', dtype=str)
lists = chk.answer.str.split(' ')
corpus_ids = set(bi3.item_id)
assert list(chk.columns) == ['query_id', 'answer']
assert len(chk) == len(bq3) and set(chk.query_id) == set(bq3.query_id) and chk.query_id.is_unique
assert chk.query_id.str.len().eq(16).all()
assert lists.map(len).le(50).all() and lists.map(lambda l: len(l) == len(set(l))).all()
assert lists.map(lambda l: all(x in corpus_ids for x in l)).all()
print('answer.csv: формат корректен;', len(chk), 'строк; кандидатов в строке:', lists.map(len).min(), '-', lists.map(len).max())
# сколько ответов изменилось относительно бейзлайна (для понимания масштаба изменений)
base_top = pd.Series([set(p) for p in predictions], index=bq.query_id)
print('средняя доля новых item_id относительно бейзлайна: %.2f' % np.mean(
    [len(set(a.split()) - base_top[q]) / 50 for q, a in zip(answer.query_id, answer.answer)]))

готово за 104 c
answer.csv: формат корректен; 2452 строк; кандидатов в строке: 50 - 50
средняя доля новых item_id относительно бейзлайна: 0.30
